<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">یک راه مستقیم برای مقدار و مشتق</h1>
<p style="text-align:right">درس 44 از 92 · چرا خروجی تبدیل را به ورودی اضافه می‌کنیم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">38-residual</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/38-residual.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">اثر مسیر جمع را در خروجی و <bdi dir="ltr">Gradient</bdi> جداگانه مشاهده کنید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: جمع</span> عضو‌به‌عضو، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">requires_grad</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۱۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر زیرلایه اصلاح صفر بسازد، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">y=x+f(x)</code> چه می‌شود؟ برای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">f(x)=2x</code> مشتق چند است و برای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">f(x)=-x</code> چه؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
x = torch.tensor([2.,-1.,3.],requires_grad=True)
print('input preserved by zero update:',x+torch.zeros_like(x))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">residual_update(x, scale)</code> را برای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">f(x)=scale*x</code> بنویسید. <bdi dir="ltr">Tensor</bdi> خروجی باید هم مقدار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> را نگه دارد و هم مسیر مشتق مستقیم آن را؛ از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">detach</code> یا ساخت <bdi dir="ltr">Tensor</bdi> تازه از عددها استفاده نکنید.</p>
</div>

In [ ]:
def residual_update(x, scale):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = residual_update(x,2.)
    if result is None: return False
    torch.testing.assert_close(result,3*x)
    grad = torch.autograd.grad(result.sum(),x)[0]
    torch.testing.assert_close(grad,torch.full_like(x,3.))
    for scale in (0.,-1.,0.5):
        z = torch.tensor([1.,4.],requires_grad=True)
        y = residual_update(z,scale)
        torch.testing.assert_close(y,(1+scale)*z)
        torch.testing.assert_close(torch.autograd.grad(y.sum(),z)[0],torch.full_like(z,1+scale))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">برای مشاهدهٔ مسیر مستقیم در بلوک واقعی، همهٔ <bdi dir="ltr">Parameter</bdi>های آن را موقتاً صفر می‌کنیم؛ جزئیات داخل بلوک را در درس‌های بعد باز می‌کنیم. فقط گزینهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">residual</code> را تغییر دهید. ورودی و وزن‌ها ثابت‌اند. نتیجهٔ این حالت کنترل‌شده را با تضمین غیرصفرماندن <bdi dir="ltr">Gradient</bdi> در هر مدل اشتباه نگیرید.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import TransformerBlock
block = TransformerBlock(ModelConfig(8,4,4,1,1,0.)).eval()
with torch.no_grad():
    for parameter in block.parameters():
        parameter.zero_()
for enabled in (False,True):
    z = torch.arange(8.).reshape(1,2,4).requires_grad_()
    y = block(z,residual=enabled)
    gradient = torch.autograd.grad(y.sum(),z)[0]
    print('residual, output, input gradient:',enabled,y.detach(),gradient)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">کد خراب <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x.detach</code> را در میان‌بر می‌گذارد. مقدار خروجی همان است، ولی یک سهم مشتق حذف می‌شود. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">intact_skip(x, update)</code> را اصلاح کنید؛ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">update</code> یک <bdi dir="ltr">Tensor</bdi> وابسته به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> است.</p>
</div>

In [ ]:
z = torch.tensor(4.,requires_grad=True)
wrong = z.detach()+2*z
wrong.backward()
print('same output, missing skip gradient:',wrong.item(),z.grad.item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def intact_skip(x, update):
    # TODO
    return None

In [ ]:
def test_repair():
    z = torch.tensor([2.,3.],requires_grad=True)
    result = intact_skip(z,2*z)
    if result is None: return False
    torch.testing.assert_close(torch.autograd.grad(result.sum(),z)[0],torch.full_like(z,3.))
    z = torch.tensor(2.,requires_grad=True)
    assert torch.autograd.grad(intact_skip(z,0*z),z)[0].item() == 1.
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">TransformerBlock</code> دو بار جمع مستقیم دارد. در <bdi dir="ltr">API</bdi> واقعی، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">residual=False</code> هر دو جمع را حذف می‌کند؛ این با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">detach</code>کردن میان‌بر یا کم‌کردن <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> از خروجی نهایی یکسان نیست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا آزمون فقط مقدار خروجی، خطای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">detach</code> در مسیر مستقیم را پیدا نمی‌کند؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/38-residual.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/38-residual.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>